# CAB ratio from Pythia/FastJet jet-pT slices

This notebook follows the `heppyyier-utils` Pythia/FastJet demo style, generates pp hard-QCD events in memory, runs the CAB Lund-prong analysis, and compares CAB spectra for jets near 100 GeV and 120 GeV.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cab_eec").is_dir():
            return path
    raise RuntimeError("Run this notebook from the CAB repository or a subdirectory.")


REPO_DIR = find_repo_dir(Path.cwd())
cab_src = REPO_DIR / "src"
if str(cab_src) not in sys.path:
    sys.path.insert(0, str(cab_src))

utils_src = REPO_DIR.parent / "heppyyier-utils" / "src"
if utils_src.is_dir() and str(utils_src) not in sys.path:
    sys.path.insert(0, str(utils_src))

from heppyyier_utils.cache import ArtifactCache, safe_token
from heppyyier_utils.pythia import PythiaConfig, create_pythia

from cab_eec.eec import EECAccumulator, eec_edges
from cab_eec.event_io import Event
from cab_eec.fastjet_backend import FastJetBackend, JetSplitter
from cab_eec.io_tables import eec_rows

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})

In [ ]:
N_EVENTS = 50000
ECM_GEV = 13_000.0
PTHAT_MIN = 70.0
RANDOM_SEED = 12345
PRINT_PYTHIA_STAT = True

JET_R = 0.4
ETA_MAX = 2.0
ETA_MARGIN = 0.05
PT_PARTICLE_MIN = 0.1

PT_BINS = {
    "pt100": (95.0, 105.0),
    "pt120": (115.0, 125.0),
}
PT_LABELS = {
    name: f"{lo:g} <= jet pT < {hi:g} GeV" for name, (lo, hi) in PT_BINS.items()
}

RATIO_NUMERATOR = "pt100"
RATIO_DENOMINATOR = "pt120"

SELECTIONS = [
    {"name": "maxkt", "mode": "max_kt"},
    {"name": "sd_z0p1", "mode": "soft_drop", "z_cut": 0.1},
    {"name": "sd_z0p2", "mode": "soft_drop", "z_cut": 0.2},
]
SELECTION_NAMES = [selection["name"] for selection in SELECTIONS]

NORMALIZATIONS = ["radiator_pt2"]

LNDR_MIN = -5.0
LNDR_MAX = 1.5
N_EEC_BINS = 60
RATIO_YLIM = (0.0, 4.0)

USE_DISK_CACHE = True
RECOMPUTE_GENERATION = False
RECOMPUTE_EEC = False

SAVE_FIGURES = False
OUTPUT_DIR = REPO_DIR / "outputs" / "pythia_fastjet_pt_ratio"
CACHE_DIR = OUTPUT_DIR / "cache"

Generate events, cluster anti-kt jets, and select Lund prongs for each jet-pT slice. The two slices are independent CAB samples with labels `pt100` and `pt120`.

With `USE_DISK_CACHE = True`, selected splitting records are written under `outputs/pythia_fastjet_pt_ratio/cache/` with a config hash in the filename and a JSON sidecar containing the full config. If the matching file already exists, Pythia generation and FastJet/Lund splitting are skipped. EEC tables are cached separately, so changing only plotting settings reuses both generated records and CAB tables.

In [ ]:
CACHE_SCHEMA_VERSION = 1


CACHE = ArtifactCache(CACHE_DIR, schema_version=CACHE_SCHEMA_VERSION, base_dir=REPO_DIR)


def compact_generation_stem(config: dict) -> str:
    pt_bins = "_".join(
        f"{name}_{lo:g}_{hi:g}" for name, (lo, hi) in config["pt_bins"].items()
    )
    selections = "_".join(selection["name"] for selection in config["selections"])
    return safe_token(
        f"nev{config['n_events']}_pthat{config['pthat_min']}_seed{config['seed']}"
        f"_R{config['jet_R']}_{pt_bins}_{selections}"
    )


def compact_eec_stem(config: dict) -> str:
    normalizations = "_".join(config["normalizations"])
    return safe_token(
        f"records{config['records_key']}_lndr{config['lndr_min']}_{config['lndr_max']}"
        f"_nb{config['n_bins']}_{normalizations}"
    )


def generation_config() -> dict:
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "generator": "pythia8_pp_hard_qcd",
        "n_events": int(N_EVENTS),
        "ecm_gev": float(ECM_GEV),
        "pthat_min": float(PTHAT_MIN),
        "seed": int(RANDOM_SEED),
        "jet_R": float(JET_R),
        "eta_max": float(ETA_MAX),
        "eta_margin": float(ETA_MARGIN),
        "pt_particle_min": float(PT_PARTICLE_MIN),
        "pt_bins": {name: [float(lo), float(hi)] for name, (lo, hi) in PT_BINS.items()},
        "selections": SELECTIONS,
    }


def generation_cache_path(config: dict | None = None) -> Path:
    config = generation_config() if config is None else config
    return CACHE.path("splitting_records", compact_generation_stem(config), config)


def pythia_to_cab_event(pythia, event_id: int) -> Event:
    px: list[float] = []
    py: list[float] = []
    pz: list[float] = []
    energy: list[float] = []

    event = pythia.event
    for index in range(event.size()):
        particle = event[index]
        if particle.isFinal() and particle.isVisible():
            px.append(float(particle.px()))
            py.append(float(particle.py()))
            pz.append(float(particle.pz()))
            energy.append(float(particle.e()))

    return Event(
        event_id=event_id,
        px=np.asarray(px, dtype=float),
        py=np.asarray(py, dtype=float),
        pz=np.asarray(pz, dtype=float),
        energy=np.asarray(energy, dtype=float),
        weight=1.0,
    )


def make_splitters(backend: FastJetBackend) -> dict[str, JetSplitter]:
    splitters = {}
    for sample_name, (pt_min, pt_max) in PT_BINS.items():
        jet_cfg = {
            "R": JET_R,
            "pt_min": pt_min,
            "pt_max": pt_max,
            "eta_max": ETA_MAX,
            "eta_margin": ETA_MARGIN,
            "pt_particle_min": PT_PARTICLE_MIN,
        }
        splitters[sample_name] = JetSplitter(backend, jet_cfg, SELECTIONS, sample_name)
    return splitters


def generate_splitting_records() -> tuple[dict[str, list], dict]:
    backend = FastJetBackend.load()
    backend_versions = backend.versions()
    splitters = make_splitters(backend)

    pythia_config = PythiaConfig.pp_hard_qcd(
        ecm=ECM_GEV,
        pthat_min=PTHAT_MIN,
        seed=RANDOM_SEED,
    )
    pythia = create_pythia(pythia_config, load=True)

    records_by_sample = {sample_name: [] for sample_name in PT_BINS}
    generated_events = 0
    visible_events = 0

    for i_event in tqdm(range(int(N_EVENTS)), desc="Pythia events", unit="event"):
        if not pythia.next():
            continue
        generated_events += 1

        event = pythia_to_cab_event(pythia, i_event)
        if len(event.px) == 0:
            continue
        visible_events += 1

        for sample_name, splitter in splitters.items():
            records_by_sample[sample_name].extend(splitter.records_from_event(event))

    if PRINT_PYTHIA_STAT:
        pythia.stat()

    run_info = {
        "attempted_events": int(N_EVENTS),
        "generated_events": int(generated_events),
        "visible_events": int(visible_events),
        "ecm_gev": float(ECM_GEV),
        "pthat_min_gev": float(PTHAT_MIN),
        "jet_R": float(JET_R),
        "backend_versions": backend_versions,
    }
    return records_by_sample, run_info


def load_or_generate_splitting_records() -> tuple[dict[str, list], dict]:
    config = generation_config()
    config_key = CACHE.digest(config)
    path = generation_cache_path(config)

    if USE_DISK_CACHE and path.exists() and not RECOMPUTE_GENERATION:
        payload = CACHE.read_pickle(path)
        run_info = dict(payload.get("run_info", {}))
        run_info.update(
            {
                "cache_status": "hit",
                "generation_config_hash": config_key,
                "generation_cache_path": CACHE.label(path),
            }
        )
        print(f"[cache hit] generated splitting records: {CACHE.label(path)}")
        return payload["records_by_sample"], run_info

    records, run_info = generate_splitting_records()
    run_info.update(
        {
            "cache_status": "miss" if USE_DISK_CACHE else "disabled",
            "generation_config_hash": config_key,
            "generation_cache_path": CACHE.label(path),
        }
    )
    if USE_DISK_CACHE:
        payload = {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "generation_config": config,
            "generation_config_hash": config_key,
            "run_info": run_info,
            "records_by_sample": records,
        }
        CACHE.write_pickle(path, payload, sidecar_exclude={"records_by_sample"})
        print(f"[cache save] generated splitting records: {CACHE.label(path)}")
    return records, run_info


records_by_sample, run_info = load_or_generate_splitting_records()
run_info

In [ ]:
def mean_or_nan(values: list[float]) -> float:
    return float(np.mean(values)) if values else float("nan")


def radiator_pt(record) -> float:
    return float(record.parent_pt) if record.parent_pt is not None else float(record.pt_a + record.pt_b)


def summarize_records(records_by_sample: dict[str, list]) -> list[dict[str, float | int | str]]:
    rows = []
    for sample_name, records in records_by_sample.items():
        pt_min, pt_max = PT_BINS[sample_name]
        for selection_name in SELECTION_NAMES:
            selected = [record for record in records if record.selection == selection_name]
            rows.append(
                {
                    "sample": sample_name,
                    "jet_pt_min": pt_min,
                    "jet_pt_max": pt_max,
                    "selection": selection_name,
                    "n_splittings": len(selected),
                    "n_jets": len({(record.event_id, record.jet_index) for record in selected}),
                    "mean_jet_pt": mean_or_nan([record.jet_pt for record in selected]),
                    "mean_radiator_pt": mean_or_nan([radiator_pt(record) for record in selected]),
                    "mean_Rg": mean_or_nan([record.delta_rg for record in selected]),
                    "mean_z": mean_or_nan([record.z for record in selected]),
                }
            )
    return rows


summary_rows = summarize_records(records_by_sample)
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    display(summary_rows)

Accumulate AA, BB, AB, and CAB EECs using the same CAB package implementation used by the file-based pipeline.

In [ ]:
def eec_analysis_config() -> dict:
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "records_key": run_info.get("generation_config_hash", CACHE.digest(generation_config())),
        "lndr_min": float(LNDR_MIN),
        "lndr_max": float(LNDR_MAX),
        "n_bins": int(N_EEC_BINS),
        "normalizations": list(NORMALIZATIONS),
        "selection_names": list(SELECTION_NAMES),
    }


def eec_cache_path(config: dict | None = None) -> Path:
    config = eec_analysis_config() if config is None else config
    return CACHE.path("eec_table", compact_eec_stem(config), config)


def compute_eec_table(records_by_sample: dict[str, list]) -> list[dict]:
    edges = eec_edges(LNDR_MIN, LNDR_MAX, N_EEC_BINS)
    rows: list[dict] = []

    for sample_name, records in records_by_sample.items():
        for normalization in NORMALIZATIONS:
            accumulators = {selection_name: EECAccumulator(edges) for selection_name in SELECTION_NAMES}
            for record in tqdm(
                records,
                desc=f"{sample_name} EEC {normalization}",
                unit="split",
                leave=False,
            ):
                accumulator = accumulators.get(record.selection)
                if accumulator is not None:
                    accumulator.fill_record(record, normalization)

            for selection_name, accumulator in accumulators.items():
                rows.extend(eec_rows(sample_name, selection_name, normalization, accumulator.result()))

    return rows


def load_or_compute_eec_table(records_by_sample: dict[str, list]) -> list[dict]:
    config = eec_analysis_config()
    config_key = CACHE.digest(config)
    path = eec_cache_path(config)

    if USE_DISK_CACHE and path.exists() and not RECOMPUTE_EEC:
        payload = CACHE.read_pickle(path)
        print(f"[cache hit] EEC table: {CACHE.label(path)}")
        return payload["eec_table"]

    rows = compute_eec_table(records_by_sample)
    if USE_DISK_CACHE:
        payload = {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "eec_config": config,
            "eec_config_hash": config_key,
            "n_rows": len(rows),
            "eec_table": rows,
        }
        CACHE.write_pickle(path, payload, sidecar_exclude={"eec_table"})
        print(f"[cache save] EEC table: {CACHE.label(path)}")
    return rows


eec_table = load_or_compute_eec_table(records_by_sample)
if pd is not None:
    display(pd.DataFrame(eec_table).head())
else:
    display(eec_table[:3])

In [ ]:
def rows_for(sample: str, selection: str, normalization: str) -> list[dict]:
    return sorted(
        [
            row
            for row in eec_table
            if row["sample"] == sample
            and row["selection"] == selection
            and row["normalization"] == normalization
        ],
        key=lambda row: row["bin_center_lndr"],
    )


def compatible_bins(first: list[dict], second: list[dict]) -> bool:
    if len(first) != len(second):
        return False
    return all(
        a["bin_lo_lndr"] == b["bin_lo_lndr"] and a["bin_hi_lndr"] == b["bin_hi_lndr"]
        for a, b in zip(first, second)
    )


def ratio_with_uncertainty(numerator, numerator_err, denominator, denominator_err):
    n = np.asarray(numerator, dtype=float)
    sn = np.asarray(numerator_err, dtype=float)
    d = np.asarray(denominator, dtype=float)
    sd = np.asarray(denominator_err, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(d != 0, n / d, np.nan)
        rel_n = np.where(n != 0, sn / n, 0.0)
        rel_d = np.where(d != 0, sd / d, 0.0)
        ratio_err = np.abs(ratio) * np.sqrt(rel_n**2 + rel_d**2)
    ratio_err = np.where(np.isfinite(ratio), ratio_err, np.nan)
    return ratio, ratio_err


def add_linear_rl_axis(ax):
    lo, hi = ax.get_xlim()
    ticks = [tick for tick in (0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4, 0.8) if lo <= np.log(tick) <= hi]
    if not ticks:
        return
    secax = ax.secondary_xaxis(
        "top",
        functions=(lambda value: np.exp(value), lambda value: np.log(np.maximum(value, 1.0e-12))),
    )
    secax.set_xticks(ticks)
    secax.set_xticklabels([f"{tick:g}" for tick in ticks])
    secax.set_xlabel("R_L")


def maybe_save(fig, stem: str) -> None:
    if not SAVE_FIGURES:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"{stem}.png", bbox_inches="tight")


def finish_lndr_axis(ax):
    ax.set_xlim(LNDR_MIN, LNDR_MAX)
    ax.set_xlabel("ln(delta R)")
    ax.grid(True, alpha=0.25)
    add_linear_rl_axis(ax)


def plot_cab_spectra(selection: str, normalization: str):
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    for sample_name in PT_BINS:
        group = rows_for(sample_name, selection, normalization)
        if not group:
            continue
        x = np.asarray([row["bin_center_lndr"] for row in group], dtype=float)
        y = np.asarray([row["cab_eec"] for row in group], dtype=float)
        err = np.asarray([row["cab_eec_err"] for row in group], dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if not np.any(mask):
            continue
        n_jets = int(group[0]["n_jets"])
        ax.step(x[mask], y[mask], where="mid", label=f"{sample_name}: {PT_LABELS[sample_name]}, n={n_jets}")
        band = mask & np.isfinite(err)
        ax.fill_between(x[band], y[band] - err[band], y[band] + err[band], step="mid", alpha=0.14)

    finish_lndr_axis(ax)
    ax.set_ylabel("CAB_EEC = AB / sqrt(AA * BB)")
    ax.set_title(f"Pythia pp CAB: {selection}, {normalization}")
    ax.legend(fontsize=8)
    maybe_save(fig, f"cab_pythia_pt_slices__{selection}__{normalization}")
    return fig


def plot_cab_ratio(selection: str, normalization: str):
    numerator_rows = rows_for(RATIO_NUMERATOR, selection, normalization)
    denominator_rows = rows_for(RATIO_DENOMINATOR, selection, normalization)
    if not numerator_rows or not denominator_rows:
        print(f"skip {selection}, {normalization}: missing numerator or denominator rows")
        return None
    if not compatible_bins(numerator_rows, denominator_rows):
        raise ValueError(f"incompatible bins for {selection}, {normalization}")

    x = np.asarray([row["bin_center_lndr"] for row in numerator_rows], dtype=float)
    ratio, ratio_err = ratio_with_uncertainty(
        [row["cab_eec"] for row in numerator_rows],
        [row["cab_eec_err"] for row in numerator_rows],
        [row["cab_eec"] for row in denominator_rows],
        [row["cab_eec_err"] for row in denominator_rows],
    )
    mask = np.isfinite(x) & np.isfinite(ratio)
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    ax.axhline(1.0, color="0.35", lw=1.0, ls="--")
    ax.step(x[mask], ratio[mask], where="mid", label=f"CAB({RATIO_NUMERATOR}) / CAB({RATIO_DENOMINATOR})")
    band = mask & np.isfinite(ratio_err)
    ax.fill_between(x[band], ratio[band] - ratio_err[band], ratio[band] + ratio_err[band], step="mid", alpha=0.18)

    finish_lndr_axis(ax)
    ax.set_ylim(*RATIO_YLIM)
    ax.set_ylabel("CAB ratio")
    ax.set_title(f"Pythia pp CAB pT-slice ratio: {selection}, {normalization}")
    ax.legend(fontsize=8)
    maybe_save(fig, f"ratio_cab_pythia__{RATIO_NUMERATOR}_over_{RATIO_DENOMINATOR}__{selection}__{normalization}")
    return fig

The ratio below is `CAB(pt100) / CAB(pt120)` with propagated uncertainties. Change `RATIO_NUMERATOR`, `RATIO_DENOMINATOR`, `SELECTIONS`, or `NORMALIZATIONS` above and rerun the analysis cells to compare other choices.

In [ ]:
for normalization in NORMALIZATIONS:
    for selection in SELECTION_NAMES:
        spectra_fig = plot_cab_spectra(selection, normalization)
        display(spectra_fig)
        plt.close(spectra_fig)

        ratio_fig = plot_cab_ratio(selection, normalization)
        if ratio_fig is not None:
            display(ratio_fig)
            plt.close(ratio_fig)